# This document focuses on analyzing and finding the answers to the Multi choice questions in Part 1 of the assignment

## Prepare libraries and load data:

In [ ]:
import pandas as pd

df_cus = pd.read_csv('../dataset1/customers.csv')
df_geo = pd.read_csv('../dataset1/geography.csv')
df_item = pd.read_csv('../dataset1/order_items.csv')
df_ord = pd.read_csv('../dataset1/orders.csv')
df_pay = pd.read_csv('../dataset1/payments.csv')
df_pduct = pd.read_csv('../dataset1/products.csv')
df_pmot = pd.read_csv('../dataset1/promotions.csv')
df_ret = pd.read_csv('../dataset1/returns.csv')
df_ship = pd.read_csv('../dataset1/shipments.csv')
df_web = pd.read_csv('../dataset1/web_traffic.csv')


Because the information about the data has already been described in the assignment, we will limit checking the data's `info()`

## Question 1: 
Among customers with more than one order, what is the approximate median number of days between two consecutive purchases (inter-order gap)? (Calculated from `orders.csv`)

In [ ]:
# Extract necessary columns
df_ques1 = df_ord[['order_id', 'customer_id', 'order_date']].copy()
# Inspect data types and missing values
df_ques1.info()
display(df_ques1.head(5))

In [ ]:
# Convert data types for calculation
df_ques1['order_date'] = pd.to_datetime(df_ques1['order_date'], format='%Y-%m-%d')
# Keep only customers with multiple orders and sort them
df_ques1 = df_ques1[df_ques1.duplicated(subset=['customer_id'], keep=False)].sort_values(by=['customer_id', 'order_date', 'order_id'])
# Check order frequency per customer
print(df_ques1['customer_id'].value_counts())

In [ ]:
# Calculate the time gap between consecutive purchases for each customer
df_ques1['inter_order_gap'] = df_ques1.groupby('customer_id')['order_date'].diff().dt.days
# Drop the first purchase of each customer (where the gap is NaN)
df_ques1 = df_ques1[df_ques1.groupby('customer_id').cumcount() > 0]
# Check the first 5 rows
display(df_ques1.head(5))

In [ ]:
df_ques1.describe()

In the descriptive statistics table, for the `inter_order_gap` column, the median value (50%) = **144** (days)
#### -> Choose C

## Question 2: 
Which product segment in `products.csv` has the highest average gross profit margin, using the formula (`price` - `cogs`)/`price`?

In [ ]:
# Extract necessary columns
df_ques2 = df_pduct[['segment', 'price', 'cogs']].copy()
# Inspect data types and missing values
df_ques2.info()
# Check the unique values and their frequencies in the segment column
print(df_ques2['segment'].value_counts())

In [ ]:
# Calculate and store the gross profit margin
df_ques2['gross_profit_margin'] = (df_ques2['price'] - df_ques2['cogs']) / df_ques2['price']
# Calculate the average gross profit margin for each segment
mean_by_segment = df_ques2.groupby('segment')['gross_profit_margin'].mean()
# Print result
print(mean_by_segment)
print(f"\nHighest segment: {mean_by_segment.idxmax()} with margin {mean_by_segment.max()}")

#### -> Choose D

## Question 3: 
Among the return records linked to products in the *Streetwear* category (joining `returns` with `products` on `product_id`), which return reason appears the most?

In [ ]:
# Extract necessary columns
df_ques3 = df_ret[['return_id', 'product_id', 'return_reason']].copy()
df_support = df_pduct[['product_id', 'category']].copy()
# Inspect data types and missing values
df_ques3.info()
df_support.info()

In [ ]:
# Merge product categories into returns using product_id
df_ques3 = df_ques3.merge(df_support, on='product_id', how='left')
# Filter for Streetwear products
df_ques3 = df_ques3[df_ques3['category'] == 'Streetwear']
# Print result
print(df_ques3['return_reason'].value_counts())
mode_reason = df_ques3['return_reason'].mode()[0]
print("\nThe most common reason for returns is:", mode_reason)

#### -> Choose B

## Question 4: 
In `web_traffic.csv`, which `traffic source` has the lowest average `bounce_rate` across all days that source appears in the `traffic_source` column?

In [ ]:
# Extract necessary columns
df_ques4 = df_web[['date','bounce_rate', 'traffic_source']].copy()
# Inspect data types and missing values
df_ques4.info()
display(df_ques4.head(5))

In [ ]:
# Convert date column to datetime format
df_ques4['date'] = pd.to_datetime(df_ques4['date'], format='%Y-%m-%d')
# Calculate the average bounce rate per traffic source
mean_by_traffic_source = df_ques4.groupby('traffic_source')['bounce_rate'].mean()
# Print result
print(mean_by_traffic_source)
print(f"\nTraffic source: {mean_by_traffic_source.idxmin()} has the lowest average bounce rate of {mean_by_traffic_source.min()}")

#### -> Choose C

## Question 5: 
What is the approximate percentage of rows in `order_items.csv` that have a promotion applied (i.e., `promo_id` is not null)?

In [ ]:
df_ques5 = df_item.copy()
# Inspect data types and missing values
df_ques5.info()

In [ ]:
percentage_apply_promotion = df_ques5['promo_id'].notna().sum() / len(df_ques5) * 100
print(f"The percentage of the promotion applied is: {percentage_apply_promotion.round(0)}%")

#### -> Choose C

## Question 6: 
In `customers.csv`, considering customers with a non-null `age_group`, which age group has the highest average number of orders per customer? (total orders / number of customers in the group)

In [ ]:
# Extract necessary columns
df_ques6 = df_ord[['customer_id', 'order_id']].copy()
df_support = df_cus[['customer_id', 'age_group']].copy()
# Inspect data types and missing values
df_ques6.info()
df_support.info()

In [ ]:
# Merge customer age group into the orders dataframe using customer_id
df_ques6 = df_ques6.merge(df_support, on='customer_id', how='left')
# Calculate average orders per customer for each age group
mean_by_age_group = df_ques6.groupby('age_group')['order_id'].count() / df_ques6.groupby('age_group')['customer_id'].nunique()
# Print result
print(mean_by_age_group)
print(f"Age group: {mean_by_age_group.idxmax()} has the highest average orders")

#### -> Choose A

## Question 7: 
Which `region` in `geography.csv` generates the highest total revenue in `sales_train.csv`?

In [ ]:
# Extract necessary columns
df_ques7 = df_ord[['order_id', 'order_date', 'zip', 'order_status']].copy()
df_support = df_pay[['order_id', 'payment_value']].copy()
df_support2 = df_geo[['zip', 'region']].copy()
# Inspect data types and missing values
df_ques7.info()
df_support.info()
df_support2.info()

In [ ]:
# Convert data types for calculation
df_ques7['order_date'] = pd.to_datetime(df_ques7['order_date'], format='%Y-%m-%d')
# Merge payment and geography data into the orders dataframe
df_ques7 = df_ques7.merge(df_support, on='order_id', how='left')
df_ques7 = df_ques7.merge(df_support2, on='zip', how='left')
# Print to check data types and detect missing data
df_ques7.info()

In [ ]:
# Calculate total revenue per region within the timeframe (04/07/2012 - 31/12/2022)
revenue = df_ques7.groupby('region')['payment_value'].sum()
# Print result
print(revenue)
print(f"\nRegion: {revenue.idxmax()} has the highest revenue")

#### -> Choose C

## Question 8: 
Among the orders with `order_status` = *cancelled* in `orders.csv`, which payment method is used the most?

In [ ]:
# Extract necessary columns
df_ques8 = df_ord[['order_id', 'order_status', 'payment_method']].copy()
# Filter for cancelled orders
df_ques8 = df_ques8[df_ques8['order_status'] == 'cancelled']
# Inspect data types and missing values
df_ques8.info()

In [ ]:
# Print result
print(df_ques8['payment_method'].value_counts())
mode_payment = df_ques8['payment_method'].mode()[0]
print("\nThe most commonly used payment method is:", mode_payment)

#### -> Choose A

## Question 9: 
Among the four product sizes (*S*, *M*, *L*, *XL*), which size has the highest return rate, defined as the number of records in `returns` divided by the number of rows in `order_items` (joined with `products` on `product_id`)?

In [ ]:
# Extract necessary columns
df_ques9_ret = df_ret[['return_id', 'order_id', 'product_id', 'return_quantity']].copy()
df_ques9_item = df_item[['order_id' ,'product_id', 'quantity']].copy()
df_support = df_pduct[['product_id', 'size']].copy()
# Inspect data types and missing values
df_ques9_ret.info()
df_ques9_item.info()
df_support.info()

In [ ]:
# Merge product size into the returns dataframe using product_id
df_ques9_ret = df_ques9_ret.merge(df_support, on='product_id', how='left')
# Merge product size into the order items dataframe using product_id
df_ques9_item = df_ques9_item.merge(df_support, on='product_id', how='left')
# Count total order items sold per size
sale = df_ques9_item.groupby('size').size()
# Count total returns per size
returns = df_ques9_ret.groupby('size').size()
# Print result
print(returns / sale)
print(f"\nThe product size with the highest return rate is: {(returns / sale).idxmax()}")

#### -> Choose A

## Question 10: 
In `payments.csv`, which installment plan has the highest average payment value per order?

In [ ]:
df_ques10 = df_pay.copy()
# Inspect data types and missing values
df_ques10.info()

In [ ]:
# Tính toán giá trị thanh toán trung bình trên mỗi đơn hàng trong tất cả các kế hoạch trả góp
average = df_pay.groupby('installments')['payment_value'].mean()
# Print result
print(average)
print(f"\nThe installment plan with the highest average payment is: {average.idxmax()}")

#### -> Choose C